# 09 — Embeddings, Evidence & Knowledge Loading (Milestone M3 exit)

**DSML stage:** modeling. The final M3 notebook — after this, the Nvidia PoC graph is complete and queryable.

1. Embed all chunks with **bge-large-en-v1.5** (local, 1024-dim) → `EvidenceSpan` nodes with vectors
2. Load resolved knowledge (notebook 08): `RiskFactor` nodes, relation edges, `Product` nodes
3. Wire provenance: `HAS_EVIDENCE` edges + `MENTIONS` edges (chunk → company, for entity-first retrieval)

**Design note — evidence for edges:** property graphs can't attach edges to edges, so relation edges carry
`evidence_chunk_ids` (list) + `evidence_quote` properties instead of a `HAS_EVIDENCE` edge; node-level claims
(`RiskFactor`) get true `HAS_EVIDENCE` edges. Retrieval joins back to `EvidenceSpan` by id either way.

**M3 exit criterion:** a Cypher traversal `Nvidia → suppliers` returns edges *with their supporting SEC text*.

In [ ]:
import hashlib
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()

chunks_df = pd.read_parquet(PROJECT_ROOT / "data/processed/chunks/nvda_chunks.parquet")
resolved_path = PROJECT_ROOT / "data/processed/extractions/nvda_extractions_resolved.jsonl"
assert resolved_path.exists(), "Run notebooks 07 and 08 first."
records = [json.loads(line) for line in resolved_path.open(encoding="utf-8")]
CANONICAL = json.loads((PROJECT_ROOT / "artifacts/canonical_entities.json").read_text())
print(f"{len(chunks_df)} chunks, {len(records)} extraction records, {len(CANONICAL)} canonical entities")

## 1. Embed chunks (CPU is fine at this scale; ~1-2 min for 439 chunks)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(os.getenv("EMBEDDING_MODEL", "Qwen/Qwen3-Embedding-0.6B"))
# Convention (Qwen3-Embedding & bge alike): passages/documents are embedded WITHOUT a prompt;
# only queries get a prompt/prefix — applied in notebooks 10/11 at retrieval time.
embed_input = (chunks_df["sub_heading"].fillna("") + "\n" + chunks_df["text"]).str.strip()
embeddings = model.encode(embed_input.tolist(), batch_size=8, show_progress_bar=True, normalize_embeddings=True)
assert embeddings.shape == (len(chunks_df), 1024)
print(f"embedded {embeddings.shape[0]} chunks")

## 2. EvidenceSpan nodes (+ FROM_SECTION containment, + MENTIONS company anchors)

In [ ]:
import re

# Ecosystem entities (canonical dict members beyond the 14) must exist as Company nodes BEFORE
# MENTIONS edges are created, or mentions of e.g. SK Hynix would silently drop.
ecosystem_rows = [
    {"cik": spec["entity_id"], "ticker": spec.get("ticker"), "name": name, "tier": "Ecosystem"}
    for name, spec in CANONICAL.items() if spec["cik"] is None and name != "Samsung"
]
with driver.session() as session:
    session.run(
        """UNWIND $rows AS row
        MERGE (c:Company {cik: row.cik})
        SET c.name = row.name, c.tier = coalesce(c.tier, row.tier), c.sec_filer = coalesce(c.sec_filer, false),
            c.ticker = coalesce(c.ticker, row.ticker)""",
        rows=ecosystem_rows,
    )
print(f"{len(ecosystem_rows)} ecosystem Company nodes merged")

# alias -> canonical entity_id for MENTIONS detection (word-boundary regex per alias)
alias_patterns = []
for name, spec in CANONICAL.items():
    for alias in {name, *spec["aliases"]}:
        alias_patterns.append((re.compile(rf"\b{re.escape(alias)}\b", re.I), spec["entity_id"]))

def mentioned_entities(text: str) -> list[int]:
    return sorted({eid for pat, eid in alias_patterns if pat.search(text)})

span_rows = [
    {
        "chunk_id": r.chunk_id, "text": r.text, "kind": r.kind,
        "section_key": f"{r.accession_no}:{r.section_id}",
        "accession_no": r.accession_no, "section_id": r.section_id,
        "sub_heading": r.sub_heading, "char_start": int(r.char_start), "char_end": int(r.char_end),
        "n_tokens": int(r.n_tokens), "source_url": r.source_url,
        "embedding": emb.tolist(), "mentions": mentioned_entities(r.text),
    }
    for r, emb in zip(chunks_df.itertuples(), embeddings)
]

with driver.session() as session:
    for i in range(0, len(span_rows), 100):  # batches keep transactions small
        session.run(
            """UNWIND $rows AS row
            MERGE (e:EvidenceSpan {chunk_id: row.chunk_id})
            SET e.text = row.text, e.kind = row.kind, e.sub_heading = row.sub_heading,
                e.char_start = row.char_start, e.char_end = row.char_end,
                e.n_tokens = row.n_tokens, e.source_url = row.source_url, e.embedding = row.embedding
            WITH e, row
            MATCH (s:FilingSection {section_key: row.section_key})
            MERGE (e)-[:FROM_SECTION]->(s)
            WITH e, row
            UNWIND row.mentions AS eid
            MATCH (c:Company {cik: eid})
            MERGE (e)-[:MENTIONS]->(c)""",
            rows=span_rows[i : i + 100],
        )
    n_spans = session.run("MATCH (:EvidenceSpan) RETURN count(*) AS n").single()["n"]
    n_mentions = session.run("MATCH (:EvidenceSpan)-[m:MENTIONS]->() RETURN count(m) AS n").single()["n"]
print(f"{n_spans} EvidenceSpan nodes, {n_mentions} MENTIONS edges")

## 3. Ecosystem Company nodes + relation edges + RiskFactor nodes + Product nodes

Canonical entities beyond the 14 (SK Hynix, Foxconn, ...) become `Company` nodes too (tier `Ecosystem`).
Relation edges are merged per `(source, type, target)` and **accumulate** evidence chunk ids across filings —
the bitemporal properties start as `Active` as of the earliest evidencing filing date (notebook 13 manages closure).

In [ ]:
chunk_meta = chunks_df.set_index("chunk_id")[["filing_date", "accession_no"]]

rel_rows, risk_rows, product_rows = [], [], []
for rec in records:
    fdate = chunk_meta.loc[rec["chunk_id"], "filing_date"]
    for rel in rec["relations"]:
        rel_rows.append({
            "src_id": CANONICAL[rel["source_canonical"]]["entity_id"],
            "tgt_id": CANONICAL[rel["target_canonical"]]["entity_id"],
            "rel_type": rel["relation"], "chunk_id": rec["chunk_id"],
            "quote": rel["evidence_quote"], "filing_date": fdate,
        })
    for risk in rec["risk_factors"]:
        risk_id = hashlib.sha1(f"{rec['chunk_id']}|{risk['summary']}".encode()).hexdigest()[:16]
        risk_rows.append({
            "risk_id": risk_id, "summary": risk["summary"], "category": risk["category"],
            "chunk_id": rec["chunk_id"], "quote": risk["evidence_quote"], "filing_date": fdate,
        })
    for prod in rec["products"]:
        product_rows.append({"name": prod["name"], "type": prod["type"], "chunk_id": rec["chunk_id"]})

# embed risk summaries for the risk vector index
if risk_rows:
    risk_embs = model.encode([r["summary"] for r in risk_rows], normalize_embeddings=True)
    for row, emb in zip(risk_rows, risk_embs):
        row["embedding"] = emb.tolist()

with driver.session() as session:
    for rel_type in {"SUPPLIES_TO", "DEPENDS_ON", "CUSTOMER_OF", "COMPETES_WITH"}:
        batch = [r for r in rel_rows if r["rel_type"] == rel_type]
        if batch:
            session.run(
                f"""UNWIND $rows AS row
                MATCH (s:Company {{cik: row.src_id}}), (t:Company {{cik: row.tgt_id}})
                MERGE (s)-[r:{rel_type}]->(t)
                ON CREATE SET r.start_date = date(row.filing_date), r.status = 'Active',
                              r.evidence_chunk_ids = [row.chunk_id], r.evidence_quote = row.quote
                ON MATCH SET r.evidence_chunk_ids = CASE WHEN row.chunk_id IN r.evidence_chunk_ids
                              THEN r.evidence_chunk_ids ELSE r.evidence_chunk_ids + row.chunk_id END,
                             r.start_date = CASE WHEN date(row.filing_date) < r.start_date
                              THEN date(row.filing_date) ELSE r.start_date END""",
                rows=batch,
            )
    if risk_rows:
        session.run(
            """UNWIND $rows AS row
            MERGE (rf:RiskFactor {risk_id: row.risk_id})
            SET rf.summary = row.summary, rf.category = row.category, rf.embedding = row.embedding
            WITH rf, row
            MATCH (e:EvidenceSpan {chunk_id: row.chunk_id})
            MERGE (rf)-[:HAS_EVIDENCE {quote: row.quote}]->(e)
            WITH rf, row
            MATCH (nvda:Company {ticker: 'NVDA'})
            MERGE (nvda)-[d:DISCLOSES_RISK]->(rf)
            ON CREATE SET d.start_date = date(row.filing_date), d.status = 'Active'""",
            rows=risk_rows,
        )
    if product_rows:
        session.run(
            """UNWIND $rows AS row
            MERGE (p:Product {name: row.name})
            SET p.type = coalesce(p.type, row.type)
            WITH p, row
            MATCH (e:EvidenceSpan {chunk_id: row.chunk_id})
            MERGE (p)-[:MENTIONED_IN]->(e)""",
            rows=product_rows,
        )

print(f"loaded: {len(rel_rows)} relation instances, {len(risk_rows)} risks, {len(product_rows)} product mentions")

## 4. M3 exit criterion — Nvidia → suppliers, with supporting SEC text

In [ ]:
M3_QUERY = """
MATCH (nvda:Company {ticker: 'NVDA'})-[r:SUPPLIES_TO|DEPENDS_ON|CUSTOMER_OF]-(other:Company)
UNWIND r.evidence_chunk_ids AS cid
MATCH (e:EvidenceSpan {chunk_id: cid})
RETURN startNode(r).name AS source, type(r) AS relation, endNode(r).name AS target,
       r.status AS status, toString(r.start_date) AS since,
       r.evidence_quote AS quote, e.source_url AS sec_url
ORDER BY relation, target
"""
with driver.session() as session:
    results = [dict(r) for r in session.run(M3_QUERY)]
m3_df = pd.DataFrame(results).drop_duplicates(subset=["source", "relation", "target"])
with pd.option_context("display.max_colwidth", 100):
    display(m3_df[["source", "relation", "target", "since", "quote"]])

In [ ]:
# --- M3 EXIT assertion cell ---
with driver.session() as session:
    n_spans = session.run("MATCH (:EvidenceSpan) RETURN count(*) AS n").single()["n"]
    vec_ok = session.run(
        """CALL db.index.vector.queryNodes('evidence_embedding', 3, $vec) YIELD node, score
        RETURN count(node) AS n""",
        vec=model.encode(["semiconductor supply constraints"], normalize_embeddings=True)[0].tolist(),
    ).single()["n"]
assert n_spans == len(chunks_df), "EvidenceSpan count mismatch"
assert vec_ok == 3, "vector index not answering queries"
assert len(m3_df) >= 3, "expected at least 3 supplier/customer/dependency edges for Nvidia"
assert m3_df["quote"].notna().all() and m3_df["sec_url"].notna().all()
partners = set(m3_df["source"]).union(m3_df["target"]) - {"Nvidia"}
assert "TSMC" in partners, f"TSMC missing from Nvidia's supply-chain neighborhood: {partners}"
driver.close()
print(f"M3 COMPLETE — Nvidia graph queryable: {len(m3_df)} evidenced supply-chain edges "
      f"touching {len(partners)} partners: {sorted(partners)}")